# Methods 02 - Group tasks: corals, ENSO and 20th-century warming

**How this works.** Your group does one task (about 20-30 minutes): Tasks 1-5. **Task 6 is an optional bonus** for groups that finish early. All the code is already written.
You only change the lines marked `# EDIT ME`, run the cells, look at the figure and discuss the three questions.

1. Run the **Setup** cell once (about 30-40 seconds). It loads the corals, the sea-surface temperature (SST) data and the 20CR reanalysis.
2. Jump to your task. Each task works on its own after Setup.
3. Change one thing at a time, re-run the plotting cell, and note what changes.

**Reminder - what coral d18O records.** The oxygen-isotope ratio (d18O) of coral skeleton goes **down** when the water gets
**warmer** (about -0.2 per mil per degree C) and also goes **down** when the water gets **fresher** (more rain, less salty).
So one coral d18O record mixes temperature and rainfall/salinity.

**Sign convention used in all tasks: coral index = -d18O.** We flip the sign of d18O, so the coral index goes **up** when the
water gets warmer or wetter, just like SST goes up when it gets warmer. Then red on a map means "warm/wet" for corals and SST alike
(the main notebook uses the same convention). Raw d18O always does the opposite: a positive index means *lower* d18O.

## Setup (run once, do not edit)

What this cell does, in words:
* reads the CoralHydro2k files and keeps the long coral d18O records (samples at least every 3 months, covering 1890-1980);
  the helper leaves out series that are **reconstructed seawater d18O** (ID ends in `_sw`): they are calculated from coral d18O and Sr/Ca, so they are not a second coral d18O record;
* for each coral: removes the slow trend (EMD detrending), averages the samples that fall in December-January-February (DJF), puts it on whole years 1900-1980, flips the sign (**coral index = -d18O**) and standardises it (mean 0, standard deviation 1);
* loads DJF SST (HadISST, 40S-40N, all longitudes), builds a Nino3.4 index, and computes the main SST pattern (EOF1) and the coral PCA;
* loads the 20CR surface temperature anomaly (12-month July-June means, so each value is centred on a DJF winter) and averages it over the tropics (30S-30N, all longitudes) and over a tropical Pacific box;
* for the bonus Task 6: builds the same kind of table for coral **Sr/Ca** (same selection and preparation; index = -Sr/Ca, because Sr/Ca also falls when the water warms).

In [ ]:
import io, contextlib, warnings
import numpy as np
import pandas as pd
import xarray as xr
import xeofs as xe
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pyleoclim as pyleo
from scipy import stats
import paleo_helpers as ph

# Hide only "future version" style warnings from the libraries (real problems, e.g. NaN warnings, stay visible)
for lib in ["pyleoclim", "pylipd", "xeofs", "xarray", "pandas", "cartopy"]:
    warnings.filterwarnings("ignore", category=FutureWarning, module=lib)
    warnings.filterwarnings("ignore", category=DeprecationWarning, module=lib)
    warnings.filterwarnings("ignore", category=PendingDeprecationWarning, module=lib)

plt.rcParams.update({"font.size": 11, "figure.dpi": 100})

def _quiet():
    """Hide progress bars and messages from the data readers."""
    stack = contextlib.ExitStack()
    stack.enter_context(contextlib.redirect_stdout(io.StringIO()))
    stack.enter_context(contextlib.redirect_stderr(io.StringIO()))
    return stack

# ---------- 1. corals ----------
with _quiet():
    df = ph.load_coral_table()
df_long = ph.select_long_d18o(df)                                 # long coral d18O, no '_sw' series
df_long = df_long.sort_values("dataSetName")                        # same order on every computer
assert df_long["dataSetName"].is_unique, "two series share a dataset name"

def coral_table(df_long, detrend=True, standardize=True, start=1900, stop=1980, min_obs=1):
    """Same steps as ph.prepare_corals, but on whole-year steps (step=1) and returned as a table:
    one row per DJF year, one column per coral. Values are the CORAL INDEX = -d18O (up = warmer/wetter)."""
    with _quiet():
        ms = pyleo.MultipleGeoSeries([ph.to_geoseries(r) for _, r in df_long.iterrows()], time_unit="year CE")
        if detrend:
            ms = ms.detrend(method="emd")
        ms = pyleo.MultipleGeoSeries([ph.djf_mean(g, min_obs=min_obs) for g in ms.series_list], time_unit="year CE")
        ms = ms.common_time(method="interp", start=start, stop=stop, step=1)
        if standardize:
            ms = ms.standardize()
    years = np.round(np.asarray(ms.series_list[0].time)).astype(int)
    return pd.DataFrame({g.label: -np.asarray(g.value) for g in ms.series_list},      # minus sign: -d18O
                        index=pd.Index(years, name="year"))

X = coral_table(df_long)                                         # coral index (-d18O), detrended + standardised (tasks 1-3)
X_raw = coral_table(df_long, detrend=False, standardize=False)    # -d18O in per mil, NOT detrended (task 4)
X_emd = coral_table(df_long, detrend=True, standardize=False)     # -d18O in per mil, detrended (task 4)

META = (df_long.set_index("dataSetName")[["geo_siteName", "geo_meanLat", "geo_meanLon", "geo_ocean"]]
        .rename(columns={"geo_siteName": "site", "geo_meanLat": "lat", "geo_meanLon": "lon", "geo_ocean": "ocean"}))
META["lon"] = META["lon"] % 360                                    # LiPD uses -180..180, the grids use 0..360
META["site"] = META["site"].str.slice(0, 32)

# ---------- 2. SST (HadISST, DJF, deg C, 40S-40N, lon 0-360; cells with gaps are NaN) ----------
sst = ph.load_sst().swap_dims(time="year")
NINO34 = dict(lat_min=-5, lat_max=5, lon_min=190, lon_max=240)    # 5S-5N, 170W-120W

def box_index(field, lat_min, lat_max, lon_min, lon_max, area_weighted=False):
    """Mean of a field over a lat/lon box, ignoring missing cells (lat and lon are ascending in our files).
    Small boxes near the equator: plain mean (as in the main notebook). Big boxes: area_weighted=True (cos(lat))."""
    box = field.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
    if area_weighted:
        return box.weighted(np.cos(np.deg2rad(box.lat))).mean(["lat", "lon"], skipna=True).to_series(), box
    return box.mean(["lat", "lon"], skipna=True).to_series(), box

nino34, _box = box_index(sst, **NINO34)
nino34 = nino34 - nino34.mean()

# SST EOF analysis (sqrt(cos(lat)) weights so every square km counts equally; NaN cells are left out)
eof = xe.single.EOF(n_modes=4, center=True)
eof.fit(sst, dim="year", weights=np.sqrt(np.cos(np.deg2rad(sst.lat))))
sst_eof1 = eof.components().sel(mode=1)
sst_pc1 = eof.scores().sel(mode=1).to_series()
sign = np.sign(np.corrcoef(sst_pc1, nino34)[0, 1])                 # make positive PC1 = El Nino
sst_eof1, sst_pc1 = sst_eof1 * sign, sst_pc1 * sign
sst_eof1 = sst_eof1 / float(np.abs(sst_eof1).max())                # scale the map so its largest value is 1
sst_var1 = 100 * float(eof.explained_variance_ratio().sel(mode=1))

# ---------- 3. coral PCA ----------
def coral_pca(Xs):
    """PCA of a coral table (years x corals). Re-standardises the columns first.
    Returns: % variance of each mode, PC1 loadings (one per coral), PC1 time series."""
    Z = (Xs - Xs.mean()) / Xs.std(ddof=0)
    U, s, Vt = np.linalg.svd(Z.values, full_matrices=False)
    pct = 100 * s**2 / np.sum(s**2)
    load = pd.Series(Vt[0], index=Xs.columns)
    pc = pd.Series(U[:, 0] * s[0], index=Xs.index)
    if np.corrcoef(pc, sst_pc1.loc[Xs.index])[0, 1] < 0:           # sign is arbitrary: orient like SST PC1 (high = El Nino)
        load, pc = -load, -pc
    return pct, load, pc

coral_pct, coral_load, coral_pc1 = coral_pca(X)

# ---------- 4. 20CR (skin temperature anomaly, K, relative to 1991-2020) ----------
# Each value is a 12-month JULY-JUNE mean. Time t (years since 1836-01-01) covers July(1836+t) to June(1837+t),
# so we label it 1837 + t: that is the DJF winter in the middle of the window (same label as HadISST and corals).
_ds = xr.open_dataset(ph.SKT_FILE, decode_times=False)
skt = _ds["skt"].assign_coords(time=(1837 + np.round(_ds["time"].values)).astype(int)).rename(time="year")
TROPICS = dict(lat_min=-30, lat_max=30, lon_min=0, lon_max=360)        # whole tropical band
PACIFIC = dict(lat_min=-30, lat_max=30, lon_min=120, lon_max=290)       # tropical Pacific (120E-70W)
T20_all = {name: box_index(skt.sel(year=slice(1900, 1980)), **bx, area_weighted=True)[0]
           for name, bx in [("tropics", TROPICS), ("Pacific", PACIFIC)]}
T20 = T20_all["tropics"]

# ---------- 4b. Sr/Ca (bonus Task 6): same selection rule and preparation as d18O ----------
def select_long(df, variable, first=1890, last=1980, max_step=0.25):
    """Copy of ph.select_long_d18o for any variable name (here "Sr/Ca")."""
    out = df[(df["paleoData_variableName"] == variable) & (df["minYear"] < first) & (df["maxYear"] > last)]
    step = out["year"].apply(lambda x: np.nanmedian(np.abs(np.diff(np.asarray(x, float)))))
    return out[step <= max_step + 1e-3].sort_values("dataSetName")

df_sr = select_long(df, "Sr/Ca")
assert df_sr["dataSetName"].is_unique, "two Sr/Ca series share a dataset name"
S = coral_table(df_sr)                          # coral_table flips the sign: S = -Sr/Ca, standardised (up = warmer)
PAIRED = sorted(set(S.columns) & set(X.columns))  # cores with BOTH proxies (same dataSetName = same coral)

# ---------- 5. small statistics tools ----------
def lag1(x):
    x = np.asarray(x, float) - np.mean(x)
    return np.corrcoef(x[:-1], x[1:])[0, 1]

def corr_test(a, b):
    """Correlation r of two yearly series (matched by year) and its p-value, using an
    'effective' number of years that accounts for year-to-year memory (lag-1 autocorrelation)."""
    yrs = a.index.intersection(b.index)
    a, b = a.loc[yrs], b.loc[yrs]
    r = np.corrcoef(a, b)[0, 1]
    n = len(yrs)
    ra, rb = lag1(a), lag1(b)
    neff = min(n, max(3, n * (1 - ra * rb) / (1 + ra * rb)))
    t = r * np.sqrt((neff - 2) / (1 - r**2))
    return r, 2 * stats.t.sf(abs(t), neff - 2), neff

def corr_map(series, field):
    """Correlation of one yearly series with a field at every grid cell, plus p-values (effective n)."""
    yrs = [y for y in field.year.values if y in series.index]
    f = field.sel(year=yrs)
    s = xr.DataArray(series.loc[yrs].values, dims="year", coords={"year": yrs})
    r = xr.corr(s, f, dim="year")
    fa = f - f.mean("year")
    r1_field = xr.corr(fa.isel(year=slice(0, -1)), fa.isel(year=slice(1, None)).assign_coords(year=fa.year[:-1]), dim="year")
    r1 = lag1(s.values)
    n = len(yrs)
    neff = (n * (1 - r1 * r1_field) / (1 + r1 * r1_field)).clip(3, n)
    t = r * np.sqrt((neff - 2) / (1 - r**2))
    p = xr.apply_ufunc(lambda tt, dd: 2 * stats.t.sf(np.abs(tt), dd - 2), t, neff)
    return r, p

def pattern_corr(a, b):
    """Area-weighted correlation between two maps (only cells valid in both)."""
    ok = a.notnull() & b.notnull()
    w = (np.cos(np.deg2rad(a.lat)) * xr.ones_like(a)).where(ok)
    a, b = a.where(ok), b.where(ok)
    am, bm = (a * w).sum() / w.sum(), (b * w).sum() / w.sum()
    return float(((a - am) * (b - bm) * w).sum() / np.sqrt(((a - am)**2 * w).sum() * ((b - bm)**2 * w).sum()))

_warned = []
def add_land(ax, **kw):
    """Grey land and coastlines. If the map files cannot be downloaded (offline), warn once and carry on."""
    try:
        ax.add_feature(cfeature.LAND, color="0.85", zorder=kw.get("zorder", 1))
        ax.coastlines(lw=0.5, zorder=kw.get("zorder", 1) + 1)
    except Exception as err:
        if not _warned:
            print("Note: coastlines could not be drawn (no internet?). The data are fine.", type(err).__name__)
            _warned.append(1)

def cyclic(da):
    """Repeat the first longitude at the end, so maps have no gap at the 0/360 seam."""
    from cartopy.util import add_cyclic_point
    vals, lon = add_cyclic_point(da.values, coord=da.lon.values)
    return lon, da.lat.values, vals

# ---------- spectra (Task 5): the main notebook's functions (section 2g), with optional knobs added ----------
# With the default arguments the arithmetic is identical to the main notebook, so the numbers agree.
from scipy.signal import periodogram
N_SIM = 1000                                   # number of red-noise series for the significance test

def spectrum(x, window="hann", detrend="linear"):
    # periodogram of an annual series (Hann window, linear detrend); returns period (yr) and power
    f, power = periodogram(x, fs=1.0, window=window, detrend=detrend, axis=-1)
    return 1 / f[1:], power[..., 1:]           # drop frequency 0 (infinite period)

def red_noise_95(x, seed=42, window="hann", detrend="linear", n_sim=None):
    # pointwise 95th percentile of the spectra of n_sim AR(1) series with the same lag-1 autocorrelation
    n_sim = N_SIM if n_sim is None else n_sim
    rng = np.random.default_rng(seed)          # fixed seed: same answer every time, whatever the call order
    a = np.corrcoef(x[:-1], x[1:])[0, 1]
    noise = rng.standard_normal((n_sim, len(x)))
    sims = np.zeros_like(noise)
    sims[:, 0] = noise[:, 0]
    for t in range(1, len(x)):
        sims[:, t] = a * sims[:, t - 1] + np.sqrt(1 - a**2) * noise[:, t]
    sims = sims / sims.std(axis=1, keepdims=True) * x.std()     # same variance as the data
    sim_power = spectrum(sims, window, detrend)[1]
    return a, np.percentile(sim_power, 95, axis=0), sim_power

def standardise(x):
    return (x - x.mean()) / x.std()

def trend_per_century(s):
    fit = stats.linregress(s.index.values.astype(float), s.values)
    return 100 * fit.slope, fit.pvalue

print(f"Corals: {X.shape[1]} records, DJF years {X.index[0]}-{X.index[-1]} ({len(X)} years)")
print("  by ocean:", META["ocean"].value_counts().to_dict())
print(f"SST: {sst.sizes['year']} DJF years {int(sst.year[0])}-{int(sst.year[-1])};",
      f"Nino3.4 box uses {int(_box.notnull().all('year').sum())} of {_box.isel(year=0).size} grid cells")
print(f"SST EOF1 explains {sst_var1:.0f}% of SST variance; r(SST PC1, Nino3.4) = {np.corrcoef(sst_pc1, nino34)[0, 1]:.2f}")
print(f"Coral PC1 explains {coral_pct[0]:.1f}% of coral variance; r(coral PC1, Nino3.4) = {np.corrcoef(coral_pc1, nino34.loc[X.index])[0, 1]:.2f}")
print(f"Sign check: r(coral PC1, SST PC1) = {corr_test(coral_pc1, sst_pc1)[0]:+.2f} (positive, as it should be: PC1 high = El Nino-like)")
print(f"Sr/Ca: {S.shape[1]} records ({len(PAIRED)} cores also have d18O)")
print(f"20CR tropical mean 30S-30N (July-June means, labelled by their DJF): years {T20.index[0]}-{T20.index[-1]};",
      f"r with DJF Nino3.4 = {corr_test(T20, nino34)[0]:.2f}")

---
## Task 1 - Pick a coral: does it see El Nino?

**Goal.** Choose one coral, see where it lives, and compare its coral index (-d18O) with a Nino3.4 index (DJF SST averaged over 5S-5N, 170W-120W; plain mean of the grid cells).
**Why.** El Nino warms the central/eastern Pacific and moves the rain belts; a coral records both, and the sign of the
relationship depends on where it is. **You will get** a map, a time-series plot and a correlation r.

**Step 1.** Run the next cell to see the list of corals (name, site, latitude, longitude).

In [ ]:
print(META[['site', 'lat', 'lon', 'ocean']].round(1).to_string())

**Step 2.** Type the name (first column) of the coral your group wants. Optional: change the box to make your own SST index (for example a small box around your coral).

In [ ]:
MY_CORAL = "UR00MAI01"                                          # EDIT ME: a name from the list above
BOX = dict(lat_min=-5, lat_max=5, lon_min=190, lon_max=240)      # EDIT ME (optional): Nino3.4 = 5S-5N, 190-240E

**Step 3.** Run the plotting cell (no edits needed).

In [ ]:
if MY_CORAL not in X.columns:
    raise ValueError(f"'{MY_CORAL}' is not in the list. Copy a name from the first column of the table above.")
if not (0 <= BOX["lon_min"] < BOX["lon_max"] <= 360 and -40 <= BOX["lat_min"] < BOX["lat_max"] <= 40):
    raise ValueError("The SST data cover 40S-40N; longitudes must be 0-360 east (e.g. 170W = 190). Change BOX.")
index, box = box_index(sst, **BOX)
if box.isel(year=0).size == 0 or int(box.notnull().all("year").sum()) == 0:
    raise ValueError(f"Your BOX contains 0 of {box.isel(year=0).size} usable ocean grid cells (too small, or on land). Make it bigger.")
index = (index - index.mean()) / index.std()
coral = X[MY_CORAL]
r, p, neff = corr_test(coral, index)
n_ok, n_all = int(box.notnull().all("year").sum()), box.isel(year=0).size
info = META.loc[MY_CORAL]

fig = plt.figure(figsize=(14, 4.2))
ax1 = fig.add_subplot(1, 3, 1, projection=ccrs.Robinson(central_longitude=180))
ax1.set_global()
add_land(ax1)
ax1.plot(META["lon"], META["lat"], "o", ms=4, color="0.4", transform=ccrs.PlateCarree())
ax1.plot(info["lon"], info["lat"], "*", ms=16, color="crimson", mec="k", transform=ccrs.PlateCarree())
bx = [BOX["lon_min"], BOX["lon_max"], BOX["lon_max"], BOX["lon_min"], BOX["lon_min"]]
by = [BOX["lat_min"], BOX["lat_min"], BOX["lat_max"], BOX["lat_max"], BOX["lat_min"]]
ax1.plot(bx, by, color="navy", lw=2, transform=ccrs.PlateCarree())
ax1.set_title(f"{MY_CORAL} (star) and SST box (blue)", fontsize=11)

ax2 = fig.add_subplot(1, 3, (2, 3))
ax2.plot(index.index, index.values, color="navy", lw=1.5, label="SST index (box, DJF)")
ax2.plot(coral.index, coral.values, color="crimson", lw=1.5, label=f"{MY_CORAL} coral index, -d18O (DJF)")
ax2.axhline(0, color="k", lw=0.5)
ax2.set_xlabel("Year (DJF; December counted in the next year)")
ax2.set_ylabel("Standardised units (up = warm / wet)")
ax2.set_title(f"{info['site']}  ({info['lat']:.1f}N, {info['lon']:.1f}E):  r = {r:.2f}  (p = {p:.3f})", fontsize=11)
ax2.legend(loc="upper left", fontsize=9, ncol=2)
plt.tight_layout(); plt.show()

print(f"Coral {MY_CORAL}: {info['site']}  lat {info['lat']:.1f}, lon {info['lon']:.1f} E")
print(f"SST box: {n_ok} of {n_all} grid cells have data in every year")
print(f"Correlation coral index (-d18O) vs SST index: r = {r:.2f}, p = {p:.3f} (effective number of years = {neff:.0f} of {len(coral)})")

**Discuss (no code needed).**
1. Is r positive or negative for your coral? Remember: the coral index (-d18O) goes up when the water gets warmer **or** fresher (wetter). Explain what El Nino does to the water at your site. What would the sign be for raw d18O?
2. Compare with another group that picked a coral on the other side of the Pacific. Why can two corals give opposite signs for the same El Nino?
3. Why is r never close to 1, even for the best corals? List at least two reasons (think about the coral, the measurements and the SST data).

---
## Task 2 - Teleconnection map: what does the coral network "see"?

**Goal.** Correlate the coral PC1 (the main shared signal of all corals) with SST at every grid cell and map it.
Compare that map with the main SST pattern (SST EOF1). Corals are used as coral index (-d18O), so a red coral dot means
"this coral gets warmer/wetter when PC1 is high", like red SST. **Why.** If the corals record ENSO, the map should look like ENSO.
**You will get** two maps and a "pattern correlation" number saying how similar they are.

In [ ]:
TARGET = "PC1"      # EDIT ME: "PC1" (all corals together) or one coral name from Task 1, e.g. "UR00MAI01"
ALPHA = 0.05        # EDIT ME (optional): significance level; cells with p > ALPHA get grey dots

In [ ]:
if TARGET != "PC1" and TARGET not in X.columns:
    raise ValueError(f"'{TARGET}' is not a coral name. Use \"PC1\" or a name from the Task 1 table.")
series = coral_pc1 if TARGET == "PC1" else X[TARGET]
rmap, pmap = corr_map(series, sst)
pc_ = pattern_corr(rmap, sst_eof1)

proj, pc = ccrs.PlateCarree(central_longitude=180), ccrs.PlateCarree()       # Pacific in the middle
fig = plt.figure(figsize=(13.5, 7.2))
gs = fig.add_gridspec(2, 4, width_ratios=[1, 0.015, 0.075, 0.015], wspace=0.04, hspace=0.25)
axs = [fig.add_subplot(gs[i, 0], projection=proj) for i in range(2)]
cax_r, cax_l, cax_e = fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 3]), fig.add_subplot(gs[1, 1])
levels = np.linspace(-1, 1, 21)                                                 # one symmetric scale for both maps
im = axs[0].contourf(*cyclic(rmap), levels=levels, cmap="RdBu_r", transform=pc, extend="both")
yy, xx = np.meshgrid(sst.lat, sst.lon, indexing="ij")
ns = (pmap > ALPHA).values & rmap.notnull().values
axs[0].plot(xx[ns][::2], yy[ns][::2], ".", ms=1, color="0.3", transform=pc)
if TARGET == "PC1":
    vmax = float(np.ceil(10 * coral_load.abs().max()) / 10)
    sc = axs[0].scatter(META["lon"], META["lat"], c=coral_load, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                        s=70, edgecolor="k", linewidth=1.2, transform=pc, zorder=5)
    fig.colorbar(sc, cax=cax_l, label="coral loading (-d18O) on PC1")
else:
    axs[0].plot(META.loc[TARGET, "lon"], META.loc[TARGET, "lat"], "*", ms=18, color="k", mec="w", transform=pc, zorder=5)
    cax_l.axis("off")
fig.colorbar(im, cax=cax_r, label="r with SST")
name = "coral PC1" if TARGET == "PC1" else f"{TARGET} (-d18O)"
axs[0].set_title(f"(a) Correlation of {name} with DJF SST (grey dots: p > {ALPHA}; blank: no SST data)")
im2 = axs[1].contourf(*cyclic(sst_eof1), levels=levels, cmap="RdBu_r", transform=pc)
fig.colorbar(im2, cax=cax_e, label="EOF1 (relative units, max = 1)")
axs[1].set_title(f"(b) SST EOF1 ({sst_var1:.0f}% of SST variance), positive = El Nino")
for ax in axs:
    add_land(ax, zorder=3)
    ax.set_extent([-180, 180, -40, 40], crs=proj)
    gl = ax.gridlines(draw_labels=True, lw=0.3, color="0.5", alpha=0.5); gl.top_labels = gl.right_labels = False
for cax in (cax_r, cax_l, cax_e):                     # make the colour bars as tall as the maps
    if cax.axison:
        b, m = cax.get_position(), axs[0 if cax is not cax_e else 1].get_position()
        cax.set_position([b.x0, m.y0, b.width, m.height])
plt.show()

ocean_cells = rmap.notnull()
print(f"Target: {TARGET}")
print(f"Pattern correlation between map (a) and map (b): {pc_:.2f}")
print(f"Strongest r: {float(rmap.max()):.2f} / {float(rmap.min()):.2f}")
print(f"Ocean cells with p < {ALPHA}: {100 * float((pmap < ALPHA).where(ocean_cells).mean()):.0f}%"
      f" (by chance alone you would expect about {100 * ALPHA:.0f}%)")

**Discuss.**
1. Describe map (a): where is r strongly positive, where is it negative? Why does it look like a "horseshoe" (wedge in the east, opposite sign in the west and to the north-west/south-west)?
2. The corals sit in only a few places (dots), yet the map covers the whole tropics. How can a few corals "see" SST far away from them? What does this tell you about what PC1 records?
3. Many grid cells are "significant". Is each grid cell an independent test? Why should we be careful when we count significant cells on a map?

---
## Task 3 - Is the coral PCA stable?

**Goal.** Run the coral PCA again on parts of the data (two time periods; or with some corals left out at random) and see if PC1 stays the same.
**Why.** We have only about 30 corals and 80 years, so the answer could partly be luck.
**You will get** a table, a bar chart of "% variance" with error bars, and a plot comparing the coral loadings.

**How to read the error bars.** The % variance of each mode is only approximately known, like the margin of error of an opinion poll.
The error bars show that margin, and they get wider when there are fewer years. If the error bars of two neighbouring modes
(e.g. PC2 and PC3) overlap, those two patterns cannot be told apart and may swap places from one sample to another.

**Table columns, in plain words:**
* `PC1_pct`, `PC2_pct`, `PC3_pct`: % of the coral variance in each mode. `PC1_err`, `PC2_err`: the error bars (+-).
* `PC1_PC2_distinct`, `PC2_PC3_distinct`: True if the two % values are further apart than one error bar (the usual rule).
* `PC1_PC2_bars_apart`: True if the PC1 and PC2 error bars do not touch at all (a stricter check).
* `loading_match`: how similar the coral loadings are to the all-years PCA (1 = identical).
* `r_PC1_nino34`: how well PC1 follows El Nino (Nino3.4).

*For the curious:* this error-bar rule comes from North, G.R., Bell, T.L., Cahalan, R.F. and Moeng, F.J. (1982), Sampling errors in the estimation of empirical orthogonal functions, *Monthly Weather Review*, 110, 699–706.

Corals are used as coral index (-d18O), and PC1 is oriented so that it rises with SST PC1 (El Nino), as in Task 2.

In [ ]:
PERIODS = [(1900, 1939), (1940, 1980)]   # EDIT ME: list of (first year, last year) within 1900-1980. Try three 27-year periods.
DROP_FRACTION = 0.2                      # EDIT ME: fraction of corals removed at random (0 to 0.6)
SEEDS = [0, 1, 2, 3, 4]                  # EDIT ME: each seed = a different random choice of corals

In [ ]:
for a, b in PERIODS:
    if not (1900 <= a < b <= 1980):
        raise ValueError(f"Period ({a}, {b}) must lie inside 1900-1980, with first year < last year.")
    if b - a + 1 < 20:
        print(f"Warning: {a}-{b} has only {b - a + 1} years for {X.shape[1]} corals; the PCA will mostly fit noise.")
if not 0 <= DROP_FRACTION < 1:
    raise ValueError("DROP_FRACTION must be between 0 and 1 (e.g. 0.2).")
names = sorted(X.columns)                # sorted, so the same seed drops the same corals on every computer
runs = [("all years", X)]
runs += [(f"{a}-{b}", X.loc[a:b]) for a, b in PERIODS]
for s in SEEDS:
    keep = np.random.default_rng(s).choice(names, size=int(round((1 - DROP_FRACTION) * len(names))), replace=False)
    runs.append((f"drop {DROP_FRACTION:.0%} (seed {s})", X[sorted(keep)]))

rows, loads = [], {}
for label, Xs in runs:
    pct, load, pc1 = coral_pca(Xs)
    n = len(Xs)
    e1, e2, e3 = (pct[:3] * np.sqrt(2 / n))
    rows.append(dict(run=label, corals=Xs.shape[1], years=n,
                     PC1_pct=pct[0], PC1_err=e1, PC2_pct=pct[1], PC2_err=e2, PC3_pct=pct[2],
                     PC1_PC2_distinct=(pct[0] - pct[1]) > e1, PC1_PC2_bars_apart=(pct[0] - e1) > (pct[1] + e2),
                     PC2_PC3_distinct=(pct[1] - pct[2]) > e2,
                     loading_match=np.corrcoef(load, coral_load[load.index])[0, 1],
                     r_PC1_nino34=np.corrcoef(pc1, nino34.loc[Xs.index])[0, 1]))
    loads[label] = load
res = pd.DataFrame(rows).set_index("run")

fig, axs = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.4, 1]})
xpos = np.arange(len(res))
axs[0].bar(xpos - 0.27, res.PC1_pct, 0.27, yerr=res.PC1_err, capsize=3, color="tab:red", label="PC1")
axs[0].bar(xpos, res.PC2_pct, 0.27, yerr=res.PC2_err, capsize=3, color="tab:gray", label="PC2")
axs[0].bar(xpos + 0.27, res.PC3_pct, 0.27, yerr=res.PC3_pct * np.sqrt(2 / res.years), capsize=3, color="0.8", label="PC3")
axs[0].set_xticks(xpos); axs[0].set_xticklabels(res.index, rotation=35, ha="right", fontsize=9)
axs[0].set_ylabel("% of coral variance"); axs[0].legend(ncol=3, loc="upper center")
axs[0].set_ylim(0, 1.3 * float((res.PC1_pct + res.PC1_err).max()))
axs[0].set_title("Variance explained, with error bars (margin of error)")
for label, _ in runs[1:1 + len(PERIODS)]:
    axs[1].plot(coral_load, loads[label][coral_load.index], "o", label=label, alpha=0.8)
axs[1].axline((0, 0), slope=1, color="k", lw=0.8)
axs[1].axhline(0, color="0.6", lw=0.5); axs[1].axvline(0, color="0.6", lw=0.5)
axs[1].set_xlabel("loading, all years"); axs[1].set_ylabel("loading, sub-period")
axs[1].set_title("Coral loadings on PC1: each dot is one coral"); axs[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

print(res.round(2).to_string())

**Discuss.**
1. Does PC1 keep (roughly) the same % variance, the same loadings and the same link with Nino3.4 in every run? Which change mattered more: shorter periods or fewer corals?
2. Make the periods shorter (e.g. three periods). What happens to the % variance of PC1 and PC2 and to their error bars? Why can a short record give a *higher* % variance by chance? Which pair of modes is hardest to tell apart?

---
## Task 4 - Do the corals record 20th-century warming?

**Goal.** Average the coral records and compare them with the 20CR temperature anomaly, 1900-1980
(20CR values are July-June means, labelled by the DJF winter in their middle, so the years line up with the corals).
We use the coral index **-d18O in per mil**: it goes **up** when the water warms or freshens.

**Converting temperature into coral units.** To compare temperature with the corals we convert it into the corals' units:
**expected change in -d18O = warming (deg C) x 0.2 per mil per deg C** (the `SLOPE` knob). Raw d18O falls by about 0.2 per mil
per deg C of warming, so -d18O rises by about 0.2. The black line in the plot is 20CR temperature converted in this way:
"what warming alone would give". The red line is "what the corals show".

**Why.** In Setup we *removed* the slow trend from every coral (EMD detrending), so here you can switch that step on or off.
**You will get** a time-series plot (both lines in -d18O units, with trend lines), a histogram of the trend of each coral, and the numbers.

In [ ]:
DETREND = False       # EDIT ME: False = keep the long-term trend, True = use the detrended corals from Setup
REGION = "all"        # EDIT ME: which corals: "all", "Pacific Ocean" or "Indian Ocean"
T_REGION = "tropics"  # EDIT ME: 20CR average over "tropics" (30S-30N, all longitudes) or "Pacific" (30S-30N, 120E-70W)
SLOPE = 0.2           # EDIT ME (optional): change of -d18O per degree C (per mil / deg C), e.g. 0.18 to 0.23 (raw d18O: -0.2)

In [ ]:
if not SLOPE > 0:
    raise ValueError("SLOPE must be a positive number for -d18O (try 0.2 per mil per degree C).")
if REGION not in ["all", "Pacific Ocean", "Indian Ocean"]:
    raise ValueError('REGION must be "all", "Pacific Ocean" or "Indian Ocean".')
if T_REGION not in T20_all:
    raise ValueError('T_REGION must be "tropics" or "Pacific".')
T = T20_all[T_REGION]
Xc = X_emd if DETREND else X_raw
if REGION != "all":
    Xc = Xc[[c for c in Xc.columns if META.loc[c, "ocean"] == REGION]]
anom = Xc - Xc.mean()                     # -d18O anomaly of each coral, per mil (relative to 1900-1980)
comp = anom.mean(axis=1)                  # simple average of all chosen corals

tr_T, p_T = trend_per_century(T)
tr_c, p_c = trend_per_century(comp)
each = pd.Series({c: trend_per_century(anom[c])[0] for c in anom})
expected = SLOPE * tr_T
yrs = T.index.values

T_conv = SLOPE * (T - T.mean())          # temperature converted into coral units: per mil of -d18O

fig, axs = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.8, 1]})
ax = axs[0]
ax.plot(comp.index, comp.values, color="crimson", lw=1.5, label=f"what the corals show (mean of {anom.shape[1]} corals)")
ax.plot(yrs, np.polyval(np.polyfit(comp.index, comp.values, 1), yrs), "--", color="crimson", lw=2,
        label=f"corals: trend {tr_c:+.2f} per mil/century")
ax.plot(yrs, T_conv.values, color="k", lw=1.0, alpha=0.8, label=f"what warming alone would give: 20CR ({T_REGION}) x {SLOPE}")
ax.plot(yrs, np.polyval(np.polyfit(yrs, T_conv.values, 1), yrs), "--", color="k", lw=2,
        label=f"warming alone: trend {expected:+.2f} per mil/century")
ax.set_ylabel("-d18O anomaly (per mil; up = warmer / wetter)")
ax.set_xlabel("Year")
sec = ax.secondary_yaxis("right", functions=(lambda y: y / SLOPE, lambda t: t * SLOPE))
sec.set_ylabel("20CR temperature (deg C, vs its 1900-1980 mean)")
ax.set_title(f"Corals ({'detrended' if DETREND else 'not detrended'}, {REGION}) vs 20CR ({T_REGION}), "
             f"both in -d18O units (1 deg C = {SLOPE} per mil)", fontsize=10)
ax.legend(loc="lower right", fontsize=8.5)
axs[1].hist(each, bins=np.arange(-0.8, 0.85, 0.1), color="0.6", edgecolor="k")
axs[1].axvline(0, color="k", lw=0.8)
axs[1].axvline(expected, color="k", lw=2, ls="--", label=f"what warming alone would give ({expected:+.2f})")
axs[1].axvline(tr_c, color="crimson", lw=2, label=f"what the corals show, mean ({tr_c:+.2f})")
axs[1].set_xlabel("-d18O trend of each coral (per mil per century)"); axs[1].set_ylabel("number of corals")
axs[1].legend(fontsize=9, loc="upper left")
plt.tight_layout(); plt.show()

def wiggles(s):
    """remove a straight-line trend, keep the year-to-year wiggles"""
    return s - np.polyval(np.polyfit(s.index.values, s.values, 1), s.index.values)
r_iav, p_iav, _ = corr_test(wiggles(comp), wiggles(T))
r_pc1, p_pc1, _ = corr_test(wiggles(coral_pc1), wiggles(T))
print(f"20CR warming trend ({T_REGION}): {tr_T:+.2f} K per century (p = {p_T:.3g})")
print(f"Expected -d18O trend from temperature only: {SLOPE} x {tr_T:.2f} = {expected:+.2f} per mil per century")
print(f"Observed coral-mean -d18O trend: {tr_c:+.2f} per mil per century (p = {p_c:.3g});"
      f" that is {tr_c / expected:.1f} times the temperature-only value")
print(f"Corals whose -d18O goes up (raw d18O goes down): {(each > 0).sum()} of {len(each)}")
print(f"Year-to-year wiggles (straight-line trends removed): coral-mean std {wiggles(comp).std():.3f} per mil, "
      f"20CR std {wiggles(T).std():.2f} K")
print(f"   r(coral mean, 20CR) = {r_iav:+.2f} (p = {p_iav:.2f})   <- simple average of the chosen corals")
print(f"   r(coral PC1,  20CR) = {r_pc1:+.2f} (p = {p_pc1:.2g})  <- PC1 of all corals from Setup (sign set so that high = El Nino)")

**Discuss.**
1. With `DETREND = False`: does -d18O go up over the century (i.e. do the corals get isotopically lighter)? Is the size of the trend close to what warming alone predicts? If not, what else can raise -d18O (lower raw d18O)? Try `T_REGION = "Pacific"` too.
2. Now set `DETREND = True` and re-run. What happened to the trend, and why? When is it right to detrend proxies, and when is it wrong?
3. From year to year, the simple coral average hardly follows 20CR, but the coral PC1 does. Why? (Hint: think about the sign of each coral's ENSO response in Task 1, and about what makes the tropical-mean temperature go up and down from year to year.)

---
## Task 5 - Spectral analysis: at which periods do corals and SST vary?

**Goal.** Split a time series into slow and fast wiggles and see which periods (in years) carry the most variance,
and compare up to three series side by side (by default the coral PC1 and the SST PC1).
Compare each with "red noise": random wiggles with the same year-to-year memory but no preferred period.
**Why.** ENSO comes back every 2-7 years, irregularly. If corals record ENSO, their spectrum should have more power in that band than red noise with the same memory.
**You will get** one figure with one panel per series (power against period, ENSO band shaded, grey dashed line = that series' red-noise line;
all series are standardised, so the power axis is shared and comparable) and one printed line per series.

*How to read the figure:* a peak that pokes **above the grey dashed line** is bigger than 95% of what red noise gives at that period.
Red noise itself pokes above the line *somewhere* now and then, so one peak above the line is a hint, not a proof.

*How to read the printed lines (one per panel):* the period (years) of the highest peak inside BAND; how many times higher it is than the red-noise line
(more than 1 = above the line); and the year-to-year memory (lag-1 autocorrelation) used to build the red noise.

Corals are used as coral index (-d18O), as in the other tasks.
Curious about the idea behind red noise and spectra? See the optional notebook `Spectral_analysis_primer.ipynb`.

In [ ]:
SERIES = ["coral PC1", "SST PC1"]   # EDIT ME: one name, e.g. "Nino3.4", or a list of up to three names,
                                    # e.g. ["coral PC1", "SST PC1", "UR00MAI01"]. Names: "coral PC1", "SST PC1", "Nino3.4",
                                    # or a coral name from Task 1
BAND = (2, 7)            # EDIT ME: the ENSO band, in years (between 2 and 40)
PERIOD1 = None           # EDIT ME (optional, ONE series only): e.g. (1900, 1940) to add a panel for a sub-period
PERIOD2 = None           # EDIT ME (optional, ONE series only): e.g. (1940, 1980)

In [ ]:
pick = {"coral PC1": coral_pc1, "SST PC1": sst_pc1, "Nino3.4": nino34}
names = [SERIES] if isinstance(SERIES, str) else SERIES
if not isinstance(names, (list, tuple)) or len(names) == 0:
    raise ValueError('SERIES must be one name, e.g. "coral PC1", or a list of up to three names, e.g. ["coral PC1", "SST PC1"].')
if len(names) > 3:
    raise ValueError(f"SERIES has {len(names)} names; use at most three.")
for nm in names:
    if not isinstance(nm, str):
        raise ValueError(f"{nm!r} is not a name in quotes. Example: SERIES = [\"coral PC1\", \"SST PC1\"].")
    if nm not in pick and nm not in X.columns:
        raise ValueError(f"'{nm}' is unknown. Use 'coral PC1', 'SST PC1', 'Nino3.4' or a coral name from the Task 1 table (capitals).")
if len(set(names)) < len(names):
    raise ValueError("SERIES contains the same name twice; list each series once.")
if not (isinstance(BAND, (tuple, list)) and len(BAND) == 2 and all(isinstance(v, (int, float)) for v in BAND)
        and 2 <= BAND[0] < BAND[1] <= 40):
    raise ValueError("BAND must be two numbers (shortest, longest) period in years, between 2 and 40, e.g. (2, 7).")
subs = [pp for pp in (PERIOD1, PERIOD2) if pp is not None]
if subs and len(names) > 1:
    raise ValueError("PERIOD1/PERIOD2 work with ONE series only. Set SERIES to one name (e.g. \"coral PC1\") to look at sub-periods.")
for a, b in subs:
    if not (1900 <= a < b <= 1980 and b - a + 1 >= 20):
        raise ValueError(f"Period ({a}, {b}) must lie inside 1900-1980 and be at least 20 years long.")

# one panel per (series, period): side by side for several series, stacked for one series with sub-periods
panels = [(nm, (1900, 1980)) for nm in names] + [(names[0], pp) for pp in subs]
if len(names) > 1:
    fig, axs = plt.subplots(1, len(panels), figsize=(5.2 * len(panels), 4.2), squeeze=False, sharex=True, sharey=True)
    axs = axs[0]
else:
    fig, axs = plt.subplots(len(panels), 1, figsize=(10, 3.6 * len(panels)), squeeze=False, sharex=True, sharey=True)
    axs = axs[:, 0]
for ax, (nm, (a, b)) in zip(axs, panels):
    series_ = pick[nm] if nm in pick else X[nm]
    x = standardise(np.asarray(series_.loc[a:b], float))       # same as the main notebook
    per, P = spectrum(x)
    r1, p95, Psim = red_noise_95(x)
    inband = (per >= BAND[0]) & (per <= BAND[1])
    if not inband.any():
        raise ValueError(f"No period of a {len(x)}-year series falls in BAND {BAND}; widen it "
                         f"(available periods near it: {', '.join(f'{q:.1f}' for q in per[(per > BAND[0] / 2) & (per < BAND[1] * 2)][::-1][:8])} yr).")
    colour = "#D55E00" if nm in ("SST PC1", "Nino3.4") else "#0072B2"
    ax.axvspan(BAND[0], BAND[1], color="0.9", zorder=0, label=f"band {BAND[0]}-{BAND[1]} yr")
    ax.plot(per, P, "-o", ms=3, color=colour, label=f"{nm} {a}-{b}")
    ax.plot(per, p95, "--", color="0.4", label="red-noise line (95%)")
    ax.set_xscale("log"); ax.set_xlim(2, 40)
    ax.set_xticks([2, 3, 5, 7, 10, 20, 40]); ax.set_xticklabels(["2", "3", "5", "7", "10", "20", "40"])
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
    ax.legend(fontsize=8.5, loc="upper right")
    k = np.argmax(np.where(inband, P, -np.inf))
    ax.set_title(f"{nm}, {a}-{b}: peak at {per[k]:.1f} yr ({P[k] / p95[k]:.1f} x the line)", fontsize=10.5)
    print(f"{nm} {a}-{b} ({len(x)} yr): highest peak in {BAND[0]}-{BAND[1]} yr at {per[k]:.1f} yr, "
          f"{P[k] / p95[k]:.1f} x the red-noise line; year-to-year memory (lag-1) {r1:.2f}")
    if len(x) < 60:
        print(f"   note: only {len(x)} years, so the periods are spaced coarsely; read with care.")
for ax in (axs if len(names) > 1 else axs[-1:]):
    ax.set_xlabel("period (years, log scale)")
for ax in (axs[:1] if len(names) > 1 else axs):
    ax.set_ylabel("power (standardised units)")
plt.tight_layout(); plt.show()

**Discuss.**
1. Where are the peaks for your series, and do they rise above the red-noise line? Are they inside the ENSO band?
2. Compare the panels: does the coral spectrum look like the SST spectrum? Where do they agree, where not? Why might a single coral differ?
3. Why can a spectrum of an 81-year record say almost nothing about 50-year cycles?
4. Why does the sign of the coral index (d18O or -d18O) not matter for a spectrum?

---
## Task 6 (BONUS, optional) - Same coral, two proxies: Sr/Ca vs d18O

**Goal.** Some cores were measured for both d18O and Sr/Ca. Compare how strongly each proxy follows El Nino (Nino3.4), core by core.
**Why.** Coral Sr/Ca falls when the water warms (like d18O), mainly as a temperature signal: it is largely insensitive to rain and salinity,
because seawater Sr/Ca hardly changes (it is also affected by growth rate and other biological effects). d18O feels both temperature and rain/salinity.
We use **-Sr/Ca** and **-d18O**, so both go **up** when it gets warmer (and -d18O also when it gets wetter).
**You will get** a table and plot for all paired cores, then a time-series plot for one core of your choice.

**Step 1.** Run the next cell (no edits): r with Nino3.4 for both proxies in every paired core.

In [ ]:
rows = []
for c in PAIRED:
    rows.append(dict(name=c, site=META.loc[c, "site"], ocean=META.loc[c, "ocean"],
                     r_minus_SrCa=corr_test(S[c], nino34)[0], r_minus_d18O=corr_test(X[c], nino34)[0]))
pairs = pd.DataFrame(rows).set_index("name").sort_values("r_minus_d18O")
stronger = (pairs.r_minus_d18O.abs() > pairs.r_minus_SrCa.abs())

fig, ax = plt.subplots(figsize=(8, 0.33 * len(pairs) + 1.5))
yy = np.arange(len(pairs))
ax.hlines(yy, pairs.r_minus_SrCa, pairs.r_minus_d18O, color="0.7", lw=2)
ax.plot(pairs.r_minus_d18O, yy, "o", ms=8, color=plt.cm.RdBu_r(0.9), label="-d18O")
ax.plot(pairs.r_minus_SrCa, yy, "s", ms=8, color=plt.cm.RdBu_r(0.1), label="-Sr/Ca")
ax.axvline(0, color="k", lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels([f"{n}  {pairs.loc[n, 'site'][:22]}" for n in pairs.index], fontsize=9)
ax.set_xlabel("r with Nino3.4 (DJF)"); ax.set_xlim(-1, 1); ax.legend(loc="lower right")
ax.set_title("Same cores, two proxies: correlation with El Nino")
plt.tight_layout(); plt.show()

print(pairs.round(2).to_string())
print(f"\n{len(pairs)} paired cores. -d18O has the larger |r| at {stronger.sum()} of {len(pairs)}.")
print(f"Median |r|: -Sr/Ca {pairs.r_minus_SrCa.abs().median():.2f}, -d18O {pairs.r_minus_d18O.abs().median():.2f}. "
      f"Same sign at {(np.sign(pairs.r_minus_SrCa) == np.sign(pairs.r_minus_d18O)).sum()} of {len(pairs)} cores.")

**Step 2.** Pick one core from the table and (optionally) the size of a small SST box around it.

In [ ]:
MY_CORE = "NU11PAL01"     # EDIT ME: a name from the table above (cores with both proxies)
LOCAL_BOX = 1.5           # EDIT ME (optional): half-width of the local SST box in degrees (1.5 = a 3x3 degree box)

In [ ]:
if MY_CORE not in PAIRED:
    why = "has only d18O" if MY_CORE in X.columns else ("has only Sr/Ca" if MY_CORE in S.columns else "is not in our list")
    raise ValueError(f"'{MY_CORE}' {why}. Copy a name from the paired table above (names are in capitals).")
if not 0.5 <= LOCAL_BOX <= 10:
    raise ValueError("LOCAL_BOX should be between 0.5 and 10 degrees.")
la, lo = META.loc[MY_CORE, "lat"], META.loc[MY_CORE, "lon"]
local, lbox = box_index(sst, la - LOCAL_BOX, la + LOCAL_BOX, lo - LOCAL_BOX, lo + LOCAL_BOX)
n_ok = int(lbox.notnull().all("year").sum())
z = lambda s: (s - s.mean()) / s.std()

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.plot(nino34.index, z(nino34), color="k", lw=2, label="Nino3.4 SST (DJF)")
ax.plot(X.index, X[MY_CORE], color=plt.cm.RdBu_r(0.9), lw=1.5, label="-d18O")
ax.plot(S.index, S[MY_CORE], color=plt.cm.RdBu_r(0.1), lw=1.5, label="-Sr/Ca")
if n_ok > 0:
    ax.plot(local.index, z(local), "--", color="0.5", lw=1.2, label=f"local SST (+-{LOCAL_BOX} deg box)")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Year (DJF)"); ax.set_ylabel("Standardised (up = warm / wet)")
ax.set_title(f"{MY_CORE}: {META.loc[MY_CORE, 'site']} ({la:.1f}N, {lo:.1f}E)")
ax.legend(ncol=4, fontsize=9, loc="upper left")
plt.tight_layout(); plt.show()

print(f"r with Nino3.4:   -d18O {corr_test(X[MY_CORE], nino34)[0]:+.2f}   -Sr/Ca {corr_test(S[MY_CORE], nino34)[0]:+.2f}")
if n_ok > 0:
    print(f"r with local SST: -d18O {corr_test(X[MY_CORE], local)[0]:+.2f}   -Sr/Ca {corr_test(S[MY_CORE], local)[0]:+.2f}"
          f"   ({n_ok} grid cells with data)")
else:
    print("No SST data in the local box (coast or land): make LOCAL_BOX bigger.")
print(f"r between the two proxies in this core: {corr_test(X[MY_CORE], S[MY_CORE])[0]:+.2f}")

**Discuss.**
1. Which proxy responds more strongly to El Nino, and at how many of the paired cores? Do the two proxies agree on the *sign*?
2. Why might that be? Think about what El Nino does to rain at your core, and which proxy can "see" rain.
3. What else (not climate) could make the Sr/Ca signal weaker? And would you conclude that Sr/Ca is a worse or better thermometer from this test alone?